[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giulia-open-lab/sionna-rt-course/blob/main/notebooks/02_lab_1h_student.ipynb)

# Lab: exploring radio propagation with Sionna RT

**Hands-on lab**, Master of Science (MSc) course *Radio Propagation*: 1 hour, alone or in pairs. Sionna RT is the ray-tracing (RT) module of the open-source library Sionna.

## Learning objectives

1. Apply the free-space path loss and the link budget.
2. Explain the two-ray model: nulls, break-point $d_{BP}$, path-loss exponent $\alpha$.
3. Characterise a multipath channel through its channel impulse response (CIR), power delay
   profile (PDP), delay spread and coherence bandwidth.
4. Show when the channel distorts a signal: signal bandwidth vs coherence bandwidth.

**Prerequisites:** free space and link budget, propagation mechanisms, the two-ray model, the
wideband channel (CIR, PDP, coherence bandwidth), ray tracing and ray launching.

## How this lab works

| Exercise | Topic |
|---|---|
| 0 | Setup and the scene |
| 1 | Free space: the Friis formula |
| 2 | Ground reflection: the two-ray model |
| 3 | Multipath in a street canyon |
| 4 | A signal through the channel |

- Run the setup and import cells first; then each exercise is independent. Without a graphics
  processing unit (GPU), everything runs on the central processing unit (CPU), more slowly.
- **Parameters first:** change the values at the top of an experiment cell, then run it
  (Shift+Enter). Below them is the code of the experiment: read it, and modify it if you want.
- Change one value at a time: in Exercises 1-3 your previous run is shown in grey.
- A list of values, e.g. `GROUND = ["metal", "medium_dry_ground"]`, runs a sweep.
- Each run adds a row to the **log** under the figure: use its numbers in your answers.
- In each exercise: predict when asked, run the experiments, answer in the **Your answer** cells.

## Nomenclature

| Term | Meaning |
|---|---|
| TX, RX | Transmitter, receiver. |
| Cell, small cell (micro-cell) | Cell: the area served by a base station. Small cell: a low-power base station with its antenna below the roofs, and the area it serves. |
| LoS, NLoS | Line of sight: the straight line from the TX to the RX is not blocked. Non line of sight: it is blocked. |
| Path | One way the signal goes from the TX to the RX: straight segments joined by interactions. |
| Interaction | One event where a path meets an object: a reflection, a diffraction, a diffuse scattering or a refraction. |
| Depth of a path | Its number of interactions: 0 for the direct (LoS) path, 1 for one reflection, 2 for a reflection then a diffraction, ... `MAX_DEPTH` is the largest depth searched. |
| Specular reflection (R) | On a smooth surface (a wall, the ground), with the angle of reflection equal to the angle of incidence. |
| Diffraction (D) | On an edge, e.g. the vertical corner or the roof edge of a building. |
| Diffuse scattering (S) | On a rough surface: the power is spread in many directions. |
| Refraction (T) | Transmission through a wall. |
| Ray | A direction in which the ray launcher shoots from the TX to search for paths (`NUM_RAYS` of them). In the two-ray model (Exercise 2), a ray is a path. |
| dB, dBm, dBi | Decibel: a power ratio $r$ as $10\log_{10}r$. dBm: a power relative to 1 mW. dBi: an antenna gain relative to an isotropic antenna. |
| System gain $SG$ | Received over transmitted power, in dB, antennas included: the path gain of the propagation plus the gains of the two antennas. Link budget: $P_R\,[\mathrm{dBm}]=P_T\,[\mathrm{dBm}]+SG_\mathrm{dB}$; in free space with isotropic antennas, $SG_\mathrm{dB}=-FSPL_\mathrm{dB}$, with $FSPL$ the free-space path loss. |

In [ ]:
# Setup: install Sionna RT if it is missing (Colab), select the Mitsuba variant, print versions
import importlib.util
import os
import sys
import time

SIONNA_RT_VERSION = "2.2.0"   # version this notebook was written and tested with
FORCE_CPU = False             # True: use the CPU (LLVM) variant even if a GPU is available

if importlib.util.find_spec("sionna") is None:
    if "google.colab" in sys.modules:
        print(f"Installing sionna-rt=={SIONNA_RT_VERSION}. The runtime will restart: "
              "when it has restarted, run this cell again.")
        os.system(f"pip install -q sionna-rt=={SIONNA_RT_VERSION}")
        os.kill(os.getpid(), 5)   # restart the runtime, as in the official Sionna tutorials
    else:
        raise ImportError("sionna-rt is not installed: run 'pip install -r requirements.txt' "
                          "(see the README)")
if "google.colab" in sys.modules and not os.path.exists("lab_helpers.py"):
    import urllib.request   # Colab: download the helpers of the lab from the course repository
    urllib.request.urlretrieve("https://raw.githubusercontent.com/giulia-open-lab/sionna-rt-course/"
                               "main/notebooks/lab_helpers.py", "lab_helpers.py")

import mitsuba as mi
if FORCE_CPU:
    mi.set_variant("llvm_ad_mono_polarized")   # must be set before importing sionna.rt
try:
    import sionna.rt   # selects the CUDA variant (NVIDIA's GPU platform) if a GPU is available,
                      # otherwise the LLVM variant (a compiler for the CPU)
except ImportError as err:
    if "LLVM" in str(err):
        raise RuntimeError("No GPU was found and the LLVM library needed by the CPU variant is "
                           "missing. Install it (on Ubuntu, e.g. 'sudo apt install libllvm18') or "
                           "ask the lab administrators; see the README.") from err
    raise

import drjit as dr
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from scipy.constants import epsilon_0, speed_of_light
from sionna.rt import (Camera, ITURadioMaterial, InteractionType, PathSolver, PlanarArray,
                       RadioMapSolver, Receiver, SceneObject, Transmitter, load_scene,
                       subcarrier_frequencies)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3})
SEED = 42   # seed passed to every ray-tracing call

device = "GPU (CUDA)" if mi.variant().startswith("cuda") else "CPU (LLVM)"
print(f"Python {sys.version.split()[0]} | sionna-rt {sionna.rt.__version__} | "
      f"Mitsuba {mi.__version__} | Dr.Jit {dr.__version__}")
print(f"NumPy {np.__version__} | matplotlib {matplotlib.__version__}")
print(f"Mitsuba variant: {mi.variant()}  ->  running on {device}")
if sionna.rt.__version__ != SIONNA_RT_VERSION:
    print(f"WARNING: this notebook was tested with sionna-rt {SIONNA_RT_VERSION}")

### Lab helpers

The next cell imports the helpers of the lab from `lab_helpers.py`, next to this notebook: scenes,
receivers and logs. The code of each experiment is in its exercise cell. Restarting the kernel
clears the logs.

In [ ]:
from lab_helpers import *   # scenes, receivers and logs (file lab_helpers.py)

## Exercise 0 - Setup and the scene

Run the cell: it checks the environment and shows the street canyon of Exercises 3 and 4.

In [ ]:
# Environment check: version, variant, and one line-of-sight path in an empty scene
scene = load_scene()
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.rx_array = scene.tx_array
scene.add(Transmitter(name="tx", position=[0.0, 0.0, 10.0]))
scene.add(Receiver(name="rx", position=[50.0, 0.0, 10.0]))
found = bool(PathSolver()(scene, max_depth=0).valid.numpy().any())
print(f"sionna-rt {sionna.rt.__version__}, variant {mi.variant()}, line-of-sight path found: {found}")
print("Environment OK." if found and sionna.rt.__version__ == SIONNA_RT_VERSION
      else "The environment is not as expected: re-run the setup cell and read its messages.")

# The street canyon of Exercises 3-6 seen from above: buildings, small cell (TX) and receivers
scene = make_canyon_scene(receivers=RX_POINTS)
fig, ax = plt.subplots(figsize=(10, 6.8))
for x0, x1, y0, y1, height, material in footprints(scene):
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor="0.85", edgecolor="0.4"))
    ax.text((x0 + x1)/2, (y0 + y1)/2, f"{material}\n{height:.0f} m high", ha="center", va="center", fontsize=8)
ax.plot(TX_CANYON[0], TX_CANYON[1], "r^", ms=10)
ax.annotate(f"TX ({TX_CANYON[2]:.0f} m high)", TX_CANYON[:2], textcoords="offset points", xytext=(-30, 8), color="r")
for rx_name, (x, y, _) in RX_POINTS.items():
    ax.plot(x, y, "mx", ms=9, mew=2)
    ax.annotate(rx_name, (x, y), textcoords="offset points", xytext=(5, 5), color="m", fontsize=9)
bb = scene.mi_scene.bbox()
ax.set(xlim=(float(bb.min.x), float(bb.max.x)), ylim=(float(bb.min.y), float(bb.max.y)), aspect="equal",
       xlabel="x [m]", ylabel="y [m]", title="Street canyon seen from above (concrete floor; receivers 1.5 m above it)")
ax.grid(False)
plt.show()
# ... and in 3D
scene.render(camera=cam_canyon, resolution=(640, 480), num_samples=32);

## Exercise 1 - Free space: the Friis formula

$$FSPL=\left(\frac{4\pi d}{\lambda}\right)^2=\left(\frac{4\pi d f}{c}\right)^2,\qquad
FSPL_\mathrm{dB}=20\log_{10}\frac{d}{1\ \mathrm{m}}+20\log_{10}\frac{f}{1\ \mathrm{GHz}}+32.4$$
$$SG_\mathrm{dB}=G_{TX}+G_{RX}-FSPL_\mathrm{dB}\quad(G\ \text{in dBi})$$
The RX antenna receives the power density through its effective area $A_e=g\lambda^2/4\pi$.

The experiment: a TX and 7 RX at 10 m height in an empty scene, 10 m to 1 km apart, antennas
facing each other. Dotted line: free space with isotropic antennas.

**Experiments** (start each one from the default values):
1. `FREQUENCY_GHZ = 28`.

In [ ]:
# ---------------------------------------------------------------- parameters: change these
FREQUENCY_GHZ = 3.5      # carrier frequency [GHz]
ANTENNA = "iso"          # "iso", "dipole", "hw_dipole" or "tr38901" (the same at both ends)
# ------------------------------------------------- the experiment: read it, modify it if you want
def free_space_experiment(FREQUENCY_GHZ=3.5, ANTENNA="iso"):
    distances = np.array([10.0, 20.0, 50.0, 100.0, 200.0, 500.0, 1000.0])   # TX-RX distances [m]
    name, runs = sweep(FREQUENCY_GHZ=FREQUENCY_GHZ, ANTENNA=ANTENNA)
    fig, ax = plt.subplots(figsize=(9, 4.6))
    if name is None and "Ex 1" in PREVIOUS:
        ax.semilogx(distances, PREVIOUS["Ex 1"], "o-", color="0.75", lw=4, label="previous run")
    for i, p in enumerate(runs):
        f = p["FREQUENCY_GHZ"]*1e9
        # Empty scene (free space): a TX and one RX per distance, 10 m high, facing each other
        scene = load_scene()
        scene.frequency = f
        scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern=p["ANTENNA"], polarization="V")
        scene.rx_array = scene.tx_array
        scene.add(Transmitter(name="tx", position=[0.0, 0.0, 10.0]))
        for k, d in enumerate(distances):
            scene.add(Receiver(name=f"rx{k}", position=[float(d), 0.0, 10.0], orientation=[np.pi, 0.0, 0.0]))
        a = PathSolver()(scene, max_depth=0).cir(out_type="numpy")[0][:, 0, 0, 0, :, 0]   # LoS paths
        sg = db(np.sum(np.abs(a)**2, axis=-1))                          # system gain SG [dB]
        fspl = 20*np.log10(4*np.pi*distances*f/speed_of_light)         # free-space path loss [dB]
        slope = np.polyfit(np.log10(distances), sg, 1)[0]               # [dB per decade]
        ax.semilogx(distances, sg, "o-", color=f"C{i}",
                    label=f"ray tracer: {p['FREQUENCY_GHZ']:g} GHz, {p['ANTENNA']}")
        ax.semilogx(distances, -fspl, ":", color=f"C{i}", label="free space, isotropic" if i == 0 else None)
        log("Ex 1", {"f [GHz]": f"{p['FREQUENCY_GHZ']:g}", "antenna": p["ANTENNA"],
                     "SG at 100 m [dB]": f"{sg[distances == 100][0]:.1f}", "slope [dB/decade]": f"{slope:.1f}"})
    PREVIOUS["Ex 1"] = sg
    ax.set(xlabel="distance d [m]", ylabel="system gain SG [dB]", title="Free space: system gain vs distance")
    ax.legend(fontsize=8)
    plt.show()
    show_log("Ex 1")


free_space_experiment(FREQUENCY_GHZ=FREQUENCY_GHZ, ANTENNA=ANTENNA)

**Q1.1** Which slope do you find? Why?

**Your answer:** _write here_

**Q1.2** How much lower is $SG$ at 28 GHz than at 3.5 GHz, at the same distance? Where does the
difference come from? (Try with different antenna models.)

**Your answer:** _write here_

## Exercise 2 - Ground reflection: the two-ray model

Over a flat ground the RX receives two rays: the **direct** one and the one **reflected by the
ground**. Their paths have almost the same length (so almost the same amplitude), but differ by
$\Delta r\approx2h_{TX}h_{RX}/d$: the rays arrive with a phase difference
$\beta\Delta r=4\pi h_{TX}h_{RX}/(\lambda d)$, with $\beta=2\pi/\lambda$. The ground also multiplies
the reflected ray by its reflection coefficient $\Gamma$. Adding the two rays:
$$SG\approx SG_\mathrm{FS}\left[1+|\Gamma|^2+2|\Gamma|\cos\left(\frac{4\pi h_{TX}h_{RX}}{\lambda d}-\arg\Gamma\right)\right]$$
with $SG_\mathrm{FS}$ the free-space system gain.
- **Before the break-point** $d_{BP}=4h_{TX}h_{RX}/\lambda$: the rays alternately add and cancel,
  creating peaks and nulls, so $SG$ oscillates around the free-space reference.
- **Beyond $d_{BP}$:** $SG\approx g_Tg_R(h_{TX}h_{RX})^2/d^4$. Writing $SG\propto d^{-\alpha}$, the
  path-loss exponent is $\alpha=4$.

**Setup:** TX at height `H_TX`, 400 RX at height `H_RX` from 2 m to 2 km, over a flat ground made of
`GROUND`, a material of the International Telecommunication Union (ITU). Isotropic antennas
($g_T=g_R=1$, i.e. 0 dBi); horizontal polarization, so the
reflection on the ground is transverse electric (TE).

**Output:**
- Plot: $SG$ vs distance from the ray tracer (solid) and the two-ray formula (dashed), with free
  space (dotted) and $d_{BP}$ (dash-dot).
- Log: the path-loss exponents measured on the ray-traced $SG$ (a slope of $-10\alpha$ dB per decade
  of distance): $\alpha_1$ before $d_{BP}$ (fitted from 10 m to $d_{BP}/2$) and $\alpha_2$ after it
  (fitted from $2d_{BP}$ to 2 km).

**Experiments** (start each one from the default values):
1. `GROUND = ["metal", "medium_dry_ground"]`.

In [ ]:
# ---------------------------------------------------------------- parameters: change these
GROUND = "medium_dry_ground"   # "very_dry_ground", "medium_dry_ground", "wet_ground", "concrete", "metal", ...
H_TX = 2.5                     # transmitter height [m]
H_RX = 1.5                     # receiver height [m]
FREQUENCY_GHZ = 3.5            # carrier frequency [GHz] (ITU grounds: 1 to 10 GHz)
# ------------------------------------------------- the experiment: read it, modify it if you want
def two_ray_experiment(GROUND="medium_dry_ground", H_TX=2.5, H_RX=1.5, FREQUENCY_GHZ=3.5):
    pol = "H"                                                       # horizontal antennas: TE reflection
    distances = np.logspace(np.log10(2.0), np.log10(2000.0), 400)      # horizontal distances [m]
    name, runs = sweep(GROUND=GROUND, H_TX=H_TX, H_RX=H_RX, FREQUENCY_GHZ=FREQUENCY_GHZ)
    fig, ax = plt.subplots(figsize=(10, 4.8))
    if name is None and "Ex 2" in PREVIOUS:
        ax.semilogx(distances, PREVIOUS["Ex 2"], color="0.75", lw=3, label="previous run")
    for i, p in enumerate(runs):
        f, h_tx, h_rx = p["FREQUENCY_GHZ"]*1e9, p["H_TX"], p["H_RX"]
        lam = speed_of_light/f
        # Scene: a flat ground, 10 m thick so that it reflects like a half-space
        ground = ITURadioMaterial("ground-material", p["GROUND"], thickness=10.0)
        scene = load_scene()
        scene.edit(add=[SceneObject(mi_mesh=flat_ground(-50.0, 2050.0, -100.0, 100.0), name="ground",
                                    radio_material=ground)])
        scene.frequency = f
        scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization=pol)
        scene.rx_array = scene.tx_array
        scene.add(Transmitter(name="tx", position=[0.0, 0.0, h_tx]))
        for k, d in enumerate(distances):
            scene.add(Receiver(name=f"rx{k}", position=[float(d), 0.0, h_rx]))
        # Ray tracer: direct ray + ground-reflected ray, added coherently
        paths = PathSolver()(scene, max_depth=1, los=True, specular_reflection=True, refraction=False,
                             seed=SEED)
        a = paths.cir(out_type="numpy")[0][:, 0, 0, 0, :, 0]
        sg = db(np.abs(a.sum(axis=-1))**2)                              # system gain SG [dB]
        # Two-ray formula, with the reflection coefficient Gamma of this ground
        eps_r, sigma = float(ground.relative_permittivity[0]), float(ground.conductivity[0])
        eta = eps_r - 1j*sigma/(epsilon_0*2*np.pi*f)
        r1 = np.sqrt(distances**2 + (h_tx - h_rx)**2)                   # direct ray
        r2 = np.sqrt(distances**2 + (h_tx + h_rx)**2)                   # reflected ray
        gamma = reflection_coefficient(np.arccos((h_tx + h_rx)/r2), eta, pol)
        beta = 2*np.pi/lam
        sg_two_ray = 20*np.log10(np.abs(lam/(4*np.pi)*(np.exp(-1j*beta*r1)/r1 + gamma*np.exp(-1j*beta*r2)/r2)))
        d_bp = 4*h_tx*h_rx/lam                                          # break-point
        alpha1 = path_loss_exponent(distances, sg, 10.0, d_bp/2)        # before d_BP
        alpha2 = path_loss_exponent(distances, sg, 2*d_bp, 2000.0)      # after d_BP
        if i == 0:
            sg_fs = 20*np.log10(lam/(4*np.pi*r1))                       # free space, first run
        # Plot: system gain vs distance
        label = f"{p['GROUND']}, h_TX {h_tx:g} m, h_RX {h_rx:g} m, {p['FREQUENCY_GHZ']:g} GHz"
        ax.semilogx(distances, sg, color=f"C{i}", label=f"ray tracer: {label}")
        if name is None:
            ax.semilogx(distances, sg_two_ray, "--", color="C1", lw=1, label="two-ray formula")
        ax.axvline(d_bp, color=f"C{i}", ls="-.", lw=1)
        log("Ex 2", {"ground": p["GROUND"], "h_TX [m]": f"{h_tx:g}", "h_RX [m]": f"{h_rx:g}",
                     "f [GHz]": f"{p['FREQUENCY_GHZ']:g}", "eps_r": f"{eps_r:.1f}", "sigma [S/m]": f"{sigma:.3g}",
                     "d_BP [m]": f"{d_bp:.0f}", "α1 (before d_BP)": f"{alpha1:.2f}", "α2 (after d_BP)": f"{alpha2:.2f}"})
    PREVIOUS["Ex 2"] = sg
    ax.semilogx(distances, sg_fs, "k:", lw=1, label=f"free space ({runs[0]['FREQUENCY_GHZ']:g} GHz)")
    ax.set(xlabel="horizontal distance d [m]", ylabel="system gain SG [dB]",
           ylim=(sg_fs.min() - 40, sg_fs.max() + 10),
           title="Two-ray model: system gain vs distance (dash-dot line: break-point d_BP)")
    ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    plt.show()
    show_log("Ex 2")


two_ray_experiment(GROUND=GROUND, H_TX=H_TX, H_RX=H_RX, FREQUENCY_GHZ=FREQUENCY_GHZ)

**Q2.1** Compare `metal` and `medium_dry_ground` (experiment 1): what changes in $SG$ before $d_{BP}$, and
what after it? Why?

**Your answer:** _write here_

## Exercise 3 - Multipath in a street canyon

In a street the RX receives the signal along many **paths**: the direct one (in line of sight) plus
copies reflected by the walls and the ground, diffracted by the edges of the buildings or scattered
by rough walls. Path $i$ arrives with a complex amplitude $a_i$ after a delay $\tau_i$. Together they
form the channel impulse response (CIR):
$$h(\tau)=\sum_ia_i\,\delta(\tau-\tau_i)$$
- **Power delay profile (PDP):** the powers $|a_i|^2$ vs the delays $\tau_i$. Their sum is the system
  gain, $SG=\sum_i|a_i|^2$.
- **Root-mean-square (RMS) delay spread:** how widely the power is spread in time,
  $\tau_\mathrm{rms}=\sqrt{\sum_i|a_i|^2\tau_i^2/SG-\bar\tau^2}$, with the mean delay
  $\bar\tau=\sum_i|a_i|^2\tau_i/SG$.
- **Frequency response:** $H(f)=\sum_ia_i\,e^{-j2\pi f\tau_i}$. The phases of the paths change with
  $f$, so $|H(f)|^2$ changes over frequency.
- **Coherence bandwidth:** $B_C\approx1/(5\tau_\mathrm{rms})$, the band over which $H(f)$ stays roughly
  constant.

**How Sionna finds the paths:** it shoots `NUM_RAYS` rays from the TX in all directions with the
shooting and bouncing rays (SBR) method, and
follows each one through up to `MAX_DEPTH` interactions: specular reflections (R), diffractions on
the edges (D), computed with the uniform theory of diffraction (UTD), and diffuse scatterings (S,
off by default: `SCATTERING = 0`). Then it computes each path found
exactly (image method). Refraction through the walls is off.

**Setup:** the street canyon of Exercise 0 (see its map). The small cell (TX) is at (60, 0) m on a
10 m pole, below the roofs: a micro-cell, 30 dBm at 3.5 GHz. Isotropic antennas at both ends.
`RECEIVER` puts one RX, 1.5 m high, at one of four points: RX-LoS (main street, in line of sight),
RX-NLoS (cross street, behind a building), RX-corner (just past the west end of a block) or
RX-shadow (6 m further into the shadow of the same block).

**Output:**
- Left figure: the PDP. One dot is one path: its received power on the y axis, its delay after the
  arrival of the first path on the x axis. Right figure: $|H(f)|^2$ over 100 MHz around the carrier,
  with $B_C$ shaded. Your previous run is in grey.
- Log: for each run, its settings, the number of paths, $SG$, $\tau_\mathrm{rms}$, $B_C$, the share of
  the received power carried by each type of path (LoS/R/D/S, in %) and the run time.

**Predict** (before running; the receivers are on the map of Exercise 0): which receiver do you expect to have
the larger delay spread, RX-LoS or RX-NLoS? Why?

**Your answer:** _write here_

**Experiments** (start each one from the default values):
1. `RECEIVER = "RX-LoS"`.

In [ ]:
# ---------------------------------------------------------------- parameters: change these
RECEIVER = "RX-NLoS"      # "RX-LoS", "RX-NLoS", "RX-corner" or "RX-shadow"
MAX_DEPTH = 5            # maximum number of interactions per path (0 = line of sight only)
REFLECTION = True        # specular reflections: True or False
DIFFRACTION = True       # diffraction on the edges of the buildings: True or False
SCATTERING = 0.0         # diffuse scattering of the walls: 0 (smooth) to 1 (rough)
BUILDINGS = "default"    # "default" or one ITU material for all buildings, e.g. "metal"
NUM_RAYS = 10**6         # rays shot from the transmitter (10**6 = one million)
RAY_SEED = 42            # random sequence of the rays
FREQUENCY_GHZ = 3.5      # carrier frequency [GHz]
# ------------------------------------------------- the experiment: read it, modify it if you want
def multipath_experiment(RECEIVER="RX-LoS", MAX_DEPTH=5, REFLECTION=True, DIFFRACTION=True, SCATTERING=0.0,
                         BUILDINGS="default", NUM_RAYS=10**6, RAY_SEED=42, FREQUENCY_GHZ=3.5):
    name, runs = sweep(RECEIVER=RECEIVER, MAX_DEPTH=MAX_DEPTH, REFLECTION=REFLECTION, DIFFRACTION=DIFFRACTION,
                       SCATTERING=SCATTERING, BUILDINGS=BUILDINGS, NUM_RAYS=NUM_RAYS, RAY_SEED=RAY_SEED,
                       FREQUENCY_GHZ=FREQUENCY_GHZ)
    results = []
    for p in runs:
        # Street canyon with the small cell and one receiver; Sionna finds the paths by launching
        # rays (SBR), then computes each path exactly (image method)
        scene = make_canyon_scene(p["FREQUENCY_GHZ"]*1e9, p["BUILDINGS"], p["SCATTERING"],
                                  receivers={p["RECEIVER"]: RX_POINTS[p["RECEIVER"]]})
        start = time.perf_counter()
        paths = PathSolver()(scene, max_depth=p["MAX_DEPTH"], los=True, specular_reflection=p["REFLECTION"],
                             diffuse_reflection=p["SCATTERING"] > 0, refraction=False,
                             diffraction=p["DIFFRACTION"], samples_per_src=int(p["NUM_RAYS"]), seed=p["RAY_SEED"])
        a, tau, types = paths_of_rx(paths, 0)                   # coefficients a_i, delays tau_i, mechanisms
        seconds = time.perf_counter() - start
        PREVIOUS["Ex 3 paths"] = (scene, paths)                 # for the 3D view in the next cell
        if len(a) == 0:
            print(f"{p['RECEIVER']}: no path with these settings")
            continue
        # PDP, system gain, RMS delay spread, coherence bandwidth and H(f)
        power = np.abs(a)**2
        tau = tau - tau.min()                                   # excess delays [s]
        sg = db(power.sum())
        tau_rms = np.sqrt(np.sum(power*tau**2)/power.sum() - (np.sum(power*tau)/power.sum())**2)
        b_c = 1/(5*tau_rms) if tau_rms > 0 else np.inf
        freqs = np.linspace(-50e6, 50e6, 1001)                  # 100 MHz around the carrier
        h = np.exp(-2j*np.pi*np.outer(freqs, tau)) @ a          # H(f) = sum_i a_i exp(-j 2 pi f tau_i)
        kind = np.array([mechanism(t) for t in types])          # L, R, D or S
        share = "/".join(f"{100*power[kind == m].sum()/power.sum():.0f}" for m in "LRDS")
        mech = "+".join((["R"] if p["REFLECTION"] else []) + (["D"] if p["DIFFRACTION"] else [])
                        + ([f"S{p['SCATTERING']:g}"] if p["SCATTERING"] > 0 else [])) or "LoS only"
        results.append(dict(p=p, tau=tau, power=power, kind=kind, freqs=freqs, h2=np.abs(h)**2, sg=sg,
                            tau_rms=tau_rms, b_c=b_c, n=len(a), seconds=seconds))
        log("Ex 3", {"receiver": p["RECEIVER"], "depth": str(p["MAX_DEPTH"]), "mechanisms": mech,
                     "buildings": p["BUILDINGS"], "rays": f"{int(p['NUM_RAYS']):.0e}",
                     "f [GHz]": f"{p['FREQUENCY_GHZ']:g}", "seed": str(p["RAY_SEED"]), "paths": str(len(a)),
                     "SG [dB]": f"{sg:.1f}", "rms ds [ns]": f"{tau_rms*1e9:.1f}", "B_C [MHz]": f"{b_c/1e6:.1f}",
                     "LoS/R/D/S [%]": share, "time [s]": f"{seconds:.1f}"})
    if not results:
        return
    if name is None:                    # one run: PDP and |H(f)|^2, with the previous run in grey
        r = results[0]
        fig, (ax, ax_h) = plt.subplots(1, 2, figsize=(14, 4.3))
        if "Ex 3" in PREVIOUS:
            q = PREVIOUS["Ex 3"]
            ax.plot(q["tau"]*1e9, db(q["power"]), "o", color="0.75", ms=6, label="previous run")
            ax_h.plot(q["freqs"]/1e6, db(q["h2"]), color="0.75", lw=3, label="previous run")
        top = db(r["power"]).max()
        for m, label, color in [("L", "line of sight", "C3"), ("R", "reflections only", "C0"),
                                ("D", "with a diffraction", "C2"), ("S", "with diffuse scattering", "C1")]:
            sel = r["kind"] == m
            if sel.any():
                ax.vlines(r["tau"][sel]*1e9, top - 60, db(r["power"][sel]), color=color, lw=0.8)
                ax.plot(r["tau"][sel]*1e9, db(r["power"][sel]), "o", color=color, ms=3, label=label)
        ax.set(xlabel="excess delay [ns]", ylabel="path power [dB]", ylim=(top - 60, top + 5),
               title=f"Power delay profile at {r['p']['RECEIVER']}")
        ax.legend(fontsize=8)
        ax_h.plot(r["freqs"]/1e6, db(r["h2"]), color="C0", label="this run")
        ax_h.axvspan(-r["b_c"]/2e6, r["b_c"]/2e6, color="C0", alpha=0.12, label=f"B_C = {r['b_c']/1e6:.1f} MHz")
        ax_h.set(xlabel="frequency offset from the carrier [MHz]", ylabel="|H(f)|^2 [dB]",
                 title="Channel frequency response over 100 MHz")
        ax_h.legend(fontsize=8)
        PREVIOUS["Ex 3"] = r
    else:                               # a sweep: the results vs the swept parameter
        fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
        values = [r["p"][name] for r in results]
        for ax, key, ylabel, scale in zip(axes, ["sg", "tau_rms", "n", "seconds"],
                                          ["system gain SG [dB]", "RMS delay spread [ns]", "number of paths",
                                           "run time [s]"], [1, 1e9, 1, 1]):
            ax.plot(sweep_axis(ax, name, values, ylabel), [r[key]*scale for r in results], "o-")
        fig.suptitle(f"Sweep of {name}", fontsize=10)
    fig.tight_layout()
    plt.show()
    show_log("Ex 3")


multipath_experiment(RECEIVER=RECEIVER, MAX_DEPTH=MAX_DEPTH, REFLECTION=REFLECTION, DIFFRACTION=DIFFRACTION,
                     SCATTERING=SCATTERING, BUILDINGS=BUILDINGS, NUM_RAYS=NUM_RAYS, RAY_SEED=RAY_SEED,
                     FREQUENCY_GHZ=FREQUENCY_GHZ)

In [ ]:
# Optional: 3D view of the paths of your last run of Exercise 3
scene, paths = PREVIOUS["Ex 3 paths"]
scene.render(camera=cam_canyon, paths=paths, resolution=(640, 480), num_samples=32);

**Q3.1** From the log, compare $SG$, $\tau_\mathrm{rms}$ and $B_C$ at RX-LoS and RX-NLoS. At each
receiver, is the channel narrowband or wideband for a fifth-generation New Radio (5G NR) carrier of
100 MHz? And for one 30 kHz
subcarrier of that carrier? Why?

**Your answer:** _write here_

## Exercise 4 - A signal through the channel

The output of the channel is the sum of delayed, scaled copies of the input,
$v(t)=\sum_ia_i\,u(t-\tau_i)$. A pulse of duration $T$ occupies a bandwidth $B\approx1/T$:
- $B\ll B_C$: the copies overlap almost perfectly, $v(t)\approx H_0\,u(t)$: attenuated, not distorted.
- $B\gtrsim B_C$: the delayed copies spread the pulse in time and change its shape (distortion).

The experiment sends a rectangular pulse through the ray-traced channel of RX-NLoS, the receiver of
Exercise 3. In the output of the cell:
- left: the input and output signals;
- right: the spectrum of the pulse and $|H(f)|^2$.

The log gives $SG$, $\tau_\mathrm{rms}$, $B_C$ and the **similarity**: the share of the output energy that is a scaled
copy of the input (100 % = no distortion).

**Experiments** (start each one from the default values):
1. `SIGNAL_BANDWIDTH_MHZ = [0.1, 1, 10, 100, 500, 1000]`.

In [ ]:
# ---------------------------------------------------------------- parameters: change these
RECEIVER = "RX-NLoS"            # channel of Exercise 3: "RX-LoS", "RX-NLoS", "RX-corner" or "RX-shadow"
SIGNAL_BANDWIDTH_MHZ = 1        # bandwidth B of the pulse [MHz]; its duration is T = 1/B
COHERENCE_BANDWIDTH_MHZ = None  # B_C of the channel [MHz], or None for the ray-traced value
# ------------------------------------------------- the experiment: read it, modify it if you want
def signal_experiment(RECEIVER="RX-NLoS", SIGNAL_BANDWIDTH_MHZ=1.0, COHERENCE_BANDWIDTH_MHZ=None):
    name, runs = sweep(RECEIVER=RECEIVER, SIGNAL_BANDWIDTH_MHZ=SIGNAL_BANDWIDTH_MHZ,
                       COHERENCE_BANDWIDTH_MHZ=COHERENCE_BANDWIDTH_MHZ)
    channels, results = {}, []
    for p in runs:
        # Channel: the ray-traced paths of the receiver, as in Exercise 3 (once per receiver)
        if p["RECEIVER"] not in channels:
            scene = make_canyon_scene(receivers={p["RECEIVER"]: RX_POINTS[p["RECEIVER"]]})
            paths = PathSolver()(scene, max_depth=5, los=True, specular_reflection=True, diffuse_reflection=False,
                                 refraction=False, diffraction=True, samples_per_src=10**6, seed=SEED)
            a, tau, _ = paths_of_rx(paths, 0)
            channels[p["RECEIVER"]] = (a, tau - tau.min())
        a, tau = channels[p["RECEIVER"]]
        power = np.abs(a)**2
        tau_rms = np.sqrt(np.sum(power*tau**2)/power.sum() - (np.sum(power*tau)/power.sum())**2)
        b_c = 1/(5*tau_rms)
        if p["COHERENCE_BANDWIDTH_MHZ"] is not None:            # stretch or shrink the delays to set B_C
            stretch = b_c/(p["COHERENCE_BANDWIDTH_MHZ"]*1e6)
            tau, tau_rms, b_c = tau*stretch, tau_rms*stretch, b_c/stretch
        # Input: a rectangular pulse u(t) of duration T = 1/B; output: v(t) = sum_i a_i u(t - tau_i)
        b = p["SIGNAL_BANDWIDTH_MHZ"]*1e6
        t_pulse = 1/b
        dt = min(t_pulse, tau_rms)/40                           # time step
        t = np.arange(int(np.ceil((t_pulse + tau.max())/dt)) + 2)*dt
        v = np.zeros(len(t), dtype=complex)
        for a_i, tau_i in zip(a, tau):
            v[(t >= tau_i) & (t < tau_i + t_pulse)] += a_i
        # Similarity: share of the output energy that is a scaled copy of the input
        n_pulse = max(1, int(round(t_pulse/dt)))
        window = np.concatenate([[0], np.cumsum(v)])
        corr = window[n_pulse:] - window[:-n_pulse]             # v(t) correlated with u(t - s), every s
        k = int(np.argmax(np.abs(corr)))
        similarity = 100*np.abs(corr[k])**2/(np.sum(np.abs(v)**2)*n_pulse)
        freqs = np.linspace(-1, 1, 801)*max(2*b, 3*b_c)
        h = np.exp(-2j*np.pi*np.outer(freqs, tau)) @ a          # H(f) of the channel
        results.append(dict(p=p, t=t, v=v, start=k*dt, scale=np.abs(corr[k])/n_pulse, t_pulse=t_pulse, b=b,
                            b_c=b_c, freqs=freqs, h=h, similarity=similarity))
        log("Ex 4", {"receiver": p["RECEIVER"], "SG [dB]": f"{db(power.sum()):.1f}", "B [MHz]": f"{b/1e6:g}", "T [µs]": f"{t_pulse*1e6:.3g}",
                     "B_C [MHz]": f"{b_c/1e6:.3g}", "B/B_C": f"{b/b_c:.2g}", "rms ds [ns]": f"{tau_rms*1e9:.3g}",
                     "similarity [%]": f"{similarity:.0f}"})

    def plot_pulses(ax, r):                                     # input and output in the time domain
        peak = np.abs(r["v"]).max()
        u = (r["t"] >= r["start"]) & (r["t"] < r["start"] + r["t_pulse"])
        ax.plot(r["t"]*1e6, np.abs(r["v"])/peak, color="C0", label="output |v(t)|")
        ax.plot(r["t"]*1e6, u*r["scale"]/peak, "--", color="0.35", label="input u(t), scaled and aligned")
        ax.set(xlabel="time [µs]", ylabel="normalised amplitude", ylim=(0, 1.15),
               title=f"{r['p']['RECEIVER']}: B = {r['b']/1e6:g} MHz, B_C = {r['b_c']/1e6:.3g} MHz, similarity {r['similarity']:.0f} %")
        ax.title.set_fontsize(9)

    if name is None:                    # one run: time domain and frequency domain
        r = results[0]
        fig, (ax, ax_f) = plt.subplots(1, 2, figsize=(14, 4.2))
        plot_pulses(ax, r)
        ax.legend(fontsize=8)
        ax_f.plot(r["freqs"]/1e6, db(np.abs(r["h"])**2/np.mean(np.abs(r["h"])**2)), color="C1",
                  label="channel |H(f)|^2")
        ax_f.plot(r["freqs"]/1e6, db(np.sinc(r["freqs"]*r["t_pulse"])**2 + 1e-12), color="C0",
                  label="spectrum of the pulse")
        ax_f.axvspan(-r["b"]/2e6, r["b"]/2e6, color="C0", alpha=0.12, label="signal band B")
        ax_f.axvspan(-r["b_c"]/2e6, r["b_c"]/2e6, color="C1", alpha=0.12, label="coherence bandwidth B_C")
        ax_f.set(xlabel="frequency offset [MHz]", ylabel="[dB]", ylim=(-40, 10), title="Frequency domain (normalised)")
        ax_f.legend(fontsize=8)
    else:                               # a sweep: the pulses of every run and the similarity
        panels = len(results[:8]) + 1
        fig, axes = plt.subplots(-(-panels//3), 3, figsize=(16, 3.8*(-(-panels//3))), squeeze=False)
        axes = axes.ravel()
        for ax, r in zip(axes, results[:8]):
            plot_pulses(ax, r)
        axes[0].legend(fontsize=7)
        ax = axes[panels - 1]
        ax.plot(sweep_axis(ax, name, [r["p"][name] for r in results], "similarity [%]"),
                [r["similarity"] for r in results], "o-")
        ax.set(ylim=(0, 105), title="Similarity: output energy that is a copy of the input")
        for ax in axes[panels:]:
            ax.axis("off")
    fig.tight_layout()
    plt.show()
    show_log("Ex 4")


signal_experiment(RECEIVER=RECEIVER, SIGNAL_BANDWIDTH_MHZ=SIGNAL_BANDWIDTH_MHZ,
                  COHERENCE_BANDWIDTH_MHZ=COHERENCE_BANDWIDTH_MHZ)

**Q4.1** At RX-NLoS, up to which signal bandwidth is the pulse received without distortion? Compare with $B_C$.

**Your answer:** _write here_

# How to submit

1. Complete the notebook: answer every question with numbers from your logs.
2. Save it as it is, with *File → Save* (Ctrl+S), and save often. The tables (logs) printed under the
   experiment cells are the record of your runs. Do not restart the kernel and re-run the cells before
   saving: a restart empties the logs, and each re-run cell would then print a log with only its new
   run. If the kernel restarts by itself (e.g. the connection to the virtual machine drops), your logs
   are still in the outputs under the cells: save without re-running the experiments.
3. Rename the notebook `lab1_<name_surname>[_<name_surname>].ipynb` (in the VS Code Explorer:
   right-click → *Rename*). If VS Code runs on your computer connected to the virtual machine, copy
   the file to your computer: right-click → *Download...*.
4. Email it to Matteo Bernabe, [matteo.bernabe@iteam.upv.es](mailto:matteo.bernabe@iteam.upv.es),
   with the subject `Lab 1 Sionna - <name_surname>, <name_surname>` (only your name if you worked
   alone).